# 08.11 · Solutions

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cloudops-one/training-ai-ml/blob/main/modules/08-deep-learning/11-pytorch/solutions.ipynb)

In [1]:
import time, warnings
warnings.filterwarnings("ignore")
import numpy as np
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader

torch.manual_seed(42)
if torch.cuda.is_available():
    device = "cuda"                     # NVIDIA GPU (Windows / Linux)
elif torch.backends.mps.is_available():
    device = "mps"                      # Apple Silicon GPU
else:
    device = "cpu"
print("PyTorch", torch.__version__, "· running on", device)

from pathlib import Path

def models_dir():
    """The repo-root models/ folder (git-ignored); created if missing."""
    for parent in [Path.cwd(), *Path.cwd().parents]:
        if (parent / "data").is_dir() and (parent / "modules").is_dir():
            folder = parent / "models"
            break
    else:
        folder = Path("models")
    folder.mkdir(exist_ok=True)
    return folder

from torchvision import datasets

def data_dir():
    """data/raw/ in the repo (git-ignored), or a local folder on Colab."""
    return models_dir().parent / "data" / "raw"

train_raw = datasets.FashionMNIST(data_dir(), train=True, download=True)
test_raw = datasets.FashionMNIST(data_dir(), train=False, download=True)
CLASSES = train_raw.classes
print(len(train_raw), "training photos,", len(test_raw), "test photos")

class ProductPhotos(Dataset):
    """Hands out (photo, label) pairs one at a time; DataLoader groups them into batches."""
    def __init__(self, raw):
        self.x = raw.data.float().div(255).unsqueeze(1)   # (N, 1, 28, 28): channels come FIRST in PyTorch
        self.y = raw.targets
    def __len__(self):
        return len(self.y)
    def __getitem__(self, i):
        return self.x[i], self.y[i]

train_ds, test_ds = ProductPhotos(train_raw), ProductPhotos(test_raw)
train_dl = DataLoader(train_ds, batch_size=128, shuffle=True)
test_dl = DataLoader(test_ds, batch_size=512)

class SmallCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(16, 32, kernel_size=3), nn.ReLU(), nn.MaxPool2d(2))
        self.head = nn.Sequential(
            nn.Flatten(), nn.Linear(32 * 5 * 5, 64), nn.ReLU(), nn.Linear(64, 10))
    def forward(self, x):
        return self.head(self.features(x))          # raw scores ("logits"), no softmax

def n_params(model):
    return sum(p.numel() for p in model.parameters())

def train_one_epoch(model, loader, optimizer, loss_fn):
    model.train()                                    # dropout etc. switch on
    total = 0.0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()                        # 1. forget last batch's gradients
        loss = loss_fn(model(xb), yb)                # 2. forward pass and loss
        loss.backward()                              # 3. backpropagation (08.4)
        optimizer.step()                             # 4. gradient-descent update
        total += loss.item() * len(yb)
    return total / len(loader.dataset)

@torch.no_grad()
def accuracy(model, loader):
    model.eval()                                     # dropout etc. switch off
    correct = 0
    for xb, yb in loader:
        correct += (model(xb.to(device)).argmax(1).cpu() == yb).sum().item()
    return correct / len(loader.dataset)

PyTorch 2.14.1 · running on mps


60000 training photos, 10000 test photos


## Exercise 1

| layer | parameters |
|---|---|
| Conv2d(1, 16, 3) | 3·3·1·16 + 16 = 160 |
| Conv2d(16, 32, 3) | 3·3·16·32 + 32 = 4,640 |
| Linear(800, 64) | 800·64 + 64 = 51,264 |
| Linear(64, 10) | 64·10 + 10 = 650 |
| **total** | **56,714** |

In [2]:
m = SmallCNN()
for name, p in m.named_parameters():
    print(f"{name:<20} {tuple(p.shape)!s:<16} {p.numel():>7,}")
print(f"total {n_params(m):,}")

features.0.weight    (16, 1, 3, 3)        144
features.0.bias      (16,)                 16
features.3.weight    (32, 16, 3, 3)     4,608
features.3.bias      (32,)                 32
head.1.weight        (64, 800)         51,200
head.1.bias          (64,)                 64
head.3.weight        (10, 64)             640
head.3.bias          (10,)                 10
total 56,714


## Exercise 2

In [3]:
class SmallCNNDropout(SmallCNN):
    def __init__(self):
        super().__init__()
        self.head = nn.Sequential(nn.Flatten(), nn.Dropout(0.3),
                                  nn.Linear(800, 64), nn.ReLU(), nn.Linear(64, 10))

torch.manual_seed(42)
model = SmallCNNDropout().to(device)
optimizer, loss_fn = torch.optim.Adam(model.parameters(), lr=1e-3), nn.CrossEntropyLoss()
for epoch in range(3):
    train_one_epoch(model, train_dl, optimizer, loss_fn)
print(f"test accuracy with dropout: {accuracy(model, test_dl):.3f}")

test accuracy with dropout: 0.856


`accuracy` calls `model.eval()`, which switches dropout off. In training mode dropout would randomly zero 30% of the inputs to the dense layer on every prediction, so the same photo could get different answers and accuracy would be lower than the model deserves.

## Exercise 3

In [4]:
from sklearn.metrics import confusion_matrix

@torch.no_grad()
def predictions(model, loader):
    model.eval()
    preds, trues = [], []
    for xb, yb in loader:
        preds.append(model(xb.to(device)).argmax(1).cpu()); trues.append(yb)
    return torch.cat(trues).numpy(), torch.cat(preds).numpy()

y_true, y_pred = predictions(model, test_dl)
cm = confusion_matrix(y_true, y_pred)
pairs = sorted(((cm[r, c], CLASSES[r], CLASSES[c]) for r in range(10) for c in range(10) if r != c), reverse=True)
for n, t, p in pairs[:5]:
    print(f"{n:>4}  {t} → {p}")

 217  Shirt → Coat
 176  Shirt → T-shirt/top
 138  Pullover → Coat
 128  Shirt → Pullover
  69  T-shirt/top → Dress


The same family of mistakes as in 08.8: shirts, T-shirts, pullovers and coats. The framework does not change what is hard about the data.

## Exercise 4

In [5]:
torch.manual_seed(0)
original = SmallCNN().to(device)
train_one_epoch(original, train_dl, torch.optim.Adam(original.parameters()), nn.CrossEntropyLoss())
path = models_dir() / "fashion_cnn_check.pt"
torch.save(original.state_dict(), path)

reloaded = SmallCNN().to(device)
reloaded.load_state_dict(torch.load(path, map_location=device))
_, a = predictions(original, test_dl)
_, b = predictions(reloaded, test_dl)
print("identical predictions:", (a == b).all())
path.unlink()                                       # tidy up

identical predictions: True
